# DataMate — agent code-gen trên dev (D20)

Chạy agent sinh code (`src/datamate/agent/`) với một model ≤ 15B trên **câu dev của train**, ghi replay + trace để lead calibrate trên CPU (`eval/calibrate_agent.py`).

**Yêu cầu** (xem `docs/KAGGLE.md`):
- Accelerator **GPU T4 x2**, **Internet: On**, notebook để **Private**.
- Dataset Kaggle **private** chứa thư mục `train` (có `questions.jsonl`, `answers.jsonl`, `rulebook.md`, `calendar.csv`, `data/`). **Chỉ train**, không đưa public_test lên.

**Đầu ra**: `/kaggle/working/datamate_dev_<model>.zip` (replay, trace, report, log). Tải về máy, **không commit lên GitHub** (chứa câu hỏi train + output model).

In [ ]:
# ---- cấu hình ----
REPO = "https://github.com/dinhkhoi124/DateMate.git"
BRANCH = "main"                        # agent SQL (D26): nhánh feat/sql-agent cho tới khi merge
AGENT = "codegen"                      # codegen (D20) | sql (D26, text-to-SQL trên view DuckDB)
MODEL_REPO = "Qwen/Qwen3-14B-AWQ"      # ứng viên khác: Qwen/Qwen3-8B-AWQ, Qwen/Qwen2.5-Coder-14B-Instruct-AWQ
PER_SHOP = 3                           # 24 shop dev x 3 = 72 câu (cùng tập với lần chạy trước để so sánh)
SAMPLES = 5                            # k mẫu / câu (codegen). AGENT="sql" dùng SQL_SAMPLES
SQL_SAMPLES = 8                        # k mẫu / câu cho agent SQL (baseline slide: 8, tau 0.5)
TAG = "ho1"                             # gắn vào tên zip để phân biệt các lần chạy
THINKING = False                       # True: Qwen3 thinking mode (chậm hơn nhiều, cần MAX_TOKENS lớn)
MAX_TOKENS = 1536                      # token sinh / mẫu; THINKING=True thì đặt 6144
MAX_LEN = 20480                        # prompt ~10-12k token
VLLM_VERSION = "0.10.1.1"              # bản pin của image (D17); không chạy được trên T4 thì dùng fallback
VLLM_FALLBACK = None                   # None: không thử bản khác (bản khác = replay lệch image)
TRANSFORMERS_VERSION = "4.55.4"        # vLLM 0.10.1.1 gọi tokenizer.all_special_tokens_extended, transformers 5.x đã bỏ
PY_VERSION = "3.12"                    # vLLM <= 0.10.1.1 cần Python < 3.13; kernel Kaggle đã là 3.13
RUN_SMOKE = True                       # 2 câu x 1 shop trước (tốn thêm 1 lần load model)
RUN_DEV = False                        # False: bỏ lượt dev (đã có v3) khi chỉ cần holdout
RUN_HOLDOUT = True                     # chỉ số tổng, không xem lỗi (CLAUDE.md luật 2)
RUN_BENCH = False                      # benchmark tổng hợp (D24, E2): shop lạ + rulebook đổi; xem docs/BENCHMARK.md
BENCH_SIDE = "B"                       # B: 182 cell (~5 giờ T4 + ~40 phút CPU dựng biến thể). Chỉ số tổng
HOLDOUT_PER_SHOP = 12                  # 6 shop holdout x 12 = 72 câu (cùng cỡ dev để so độ nhiễu)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!python --version

In [ ]:
# Kernel Kaggle là Python 3.13, vLLM 0.10.1.1 chỉ có wheel cho <= 3.12.
# -> venv riêng Python 3.12 bằng uv. Mọi lệnh pipeline chạy bằng PY (không phải kernel).
import subprocess
def sh(cmd):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print((r.stdout + r.stderr)[-3000:])
    return r.returncode

VENV = "/tmp/venv"
PY = f"{VENV}/bin/python"
sh("pip install -q uv")
sh(f"uv venv -q --python {PY_VERSION} {VENV}")     # uv tự tải Python 3.12 nếu máy chưa có

def vllm_ok():
    code = ("import torch, vllm; assert torch.cuda.is_available(); "
            "x = torch.ones(2, 2, device='cuda'); assert float((x @ x).sum()) == 8.0; "
            "print('python', __import__('sys').version.split()[0], 'torch', torch.__version__, "
            "'cuda', torch.version.cuda, 'vllm', vllm.__version__, 'transformers', __import__('transformers').__version__, "
            "'cap', torch.cuda.get_device_capability()); "
            "from transformers import AutoTokenizer as T; assert hasattr(T, 'from_pretrained')")
    return sh(f'{PY} -c "{code}"') == 0

sh(f"uv pip install -q --python {PY} vllm=={VLLM_VERSION} transformers=={TRANSFORMERS_VERSION}")
ok = vllm_ok()
if not ok and VLLM_FALLBACK:
    print(f"vLLM {VLLM_VERSION} không chạy được -> thử {VLLM_FALLBACK}")
    sh(f"uv pip install -q --python {PY} vllm=={VLLM_FALLBACK} transformers=={TRANSFORMERS_VERSION}")
    ok = vllm_ok()
assert ok, "vLLM không chạy được: gửi output cell này cho lead"

In [ ]:
import os
import glob
!rm -rf /kaggle/working/repo && git clone -q --depth 1 -b {BRANCH} {REPO} /kaggle/working/repo
os.chdir("/kaggle/working/repo")
!git log --oneline -1
# code trên GitHub phải khớp notebook này (sửa trên máy lead mà chưa push thì Kaggle vẫn chạy bản cũ)
STALE = "find_ingredients" not in open("src/datamate/agent/helpers.py", encoding="utf8").read()
assert not STALE, f"nhánh {BRANCH} trên GitHub là bản cũ: nhắc lead push rồi chạy lại cell này"
assert AGENT != "sql" or os.path.exists("src/datamate/agent/sql_agent.py"), f"nhánh {BRANCH} chưa có agent SQL"
# deps runtime giống image (docker/requirements.txt), giữ nguyên torch/vllm vừa cài
!uv pip install -q --python {PY} -r docker/requirements.txt transformers=={TRANSFORMERS_VERSION} 2>&1 | tail -3
!{PY} -c "import pandas, numpy, transformers; print('pandas', pandas.__version__, 'numpy', numpy.__version__, 'transformers', transformers.__version__)"

In [ ]:
# dataset train -> data/train/train (repo đọc đường dẫn này)
hits = [p for p in glob.glob("/kaggle/input/**/questions.jsonl", recursive=True)
        if os.path.exists(os.path.join(os.path.dirname(p), "answers.jsonl"))]
assert hits, "không thấy thư mục train (questions.jsonl + answers.jsonl) trong /kaggle/input"
TRAIN = os.path.dirname(sorted(hits)[0])
print("train:", TRAIN)
os.makedirs("data/train", exist_ok=True)
if not os.path.exists("data/train/train"):
    os.symlink(TRAIN, "data/train/train")
!ls data/train/train | head

In [ ]:
from huggingface_hub import snapshot_download
MODEL_NAME = MODEL_REPO.split("/")[-1]
MODEL_DIR = f"/tmp/models/{MODEL_NAME}"          # tên thư mục = model_id trong khoá replay
snapshot_download(MODEL_REPO, local_dir=MODEL_DIR)
!du -sh {MODEL_DIR}

In [ ]:
import time

import torch
NGPU = torch.cuda.device_count()
OUT = f"/kaggle/working/out_{MODEL_NAME}"
os.makedirs(OUT, exist_ok=True)
os.environ.update({
    "PYTHONPATH": "src", "PYTHONHASHSEED": "0",
    "DATAMATE_LLM_DTYPE": "float16",            # T4 không có bf16; AWQ chạy fp16
    "DATAMATE_LLM_TP": str(min(NGPU, 2)),
    "DATAMATE_LLM_MAX_LEN": str(MAX_LEN),
    "DATAMATE_LLM_GPU_UTIL": "0.90",
    # khoá replay gồm model_id: thinking on/off phải khác khoá
    "DATAMATE_LLM_MODEL_ID": MODEL_NAME + ("-think" if THINKING else ""),
    "DATAMATE_LLM_THINKING": "1" if THINKING else "0",
    "DATAMATE_AGENT_SAMPLES": str(SAMPLES),
    "DATAMATE_AGENT_SQL_SAMPLES": str(SQL_SAMPLES),
    "DATAMATE_AGENT_MAX_TOKENS": str(MAX_TOKENS),
    "VLLM_WORKER_MULTIPROC_METHOD": "spawn",
})
if torch.cuda.get_device_capability()[0] < 8:   # T4 = 7.5: engine V1 của 0.10.1.1 không hỗ trợ, dùng V0
    os.environ["VLLM_USE_V1"] = "0"
    os.environ.setdefault("VLLM_ATTENTION_BACKEND", "XFORMERS")
    os.environ["DATAMATE_LLM_PREFIX_CACHE"] = "0"   # kernel Triton prefix-prefill không biên dịch được cho sm75
print("GPUs:", NGPU, {k: v for k, v in os.environ.items() if k.startswith("DATAMATE_")})
EVAL = f"{PY} eval/run_eval.py --agent {AGENT} --llm vllm --llm-model {MODEL_DIR} --llm-replay {OUT}/replay.jsonl --trace --by source"
# agent SQL: kho few-shot (gitignored, không có trên GitHub) dựng lại từ shop dev của train, chỉ CPU
if AGENT == "sql":
    assert sh(f"{PY} eval/build_fewshot_sql.py") == 0, "dựng few-shot SQL lỗi: gửi output cell này cho lead"


# chạy lệnh dài: output đầy đủ vào file, mỗi phút in thời gian + số prompt đã xong (đếm dòng replay)
def run_live(cmd, logfile, replay=f"{OUT}/replay.jsonl", show=None, every=60):
    def n_replay():
        try:
            return sum(1 for _ in open(replay, encoding="utf8"))
        except FileNotFoundError:
            return 0
    n0, t0 = n_replay(), time.time()
    print("$", cmd, flush=True)
    with open(logfile, "w") as lf:
        p = subprocess.Popen(cmd, shell=True, stdout=lf, stderr=subprocess.STDOUT)
        while p.poll() is None:
            time.sleep(every)
            extra = ""
            if show:   # dòng tiến độ do chính lệnh in (vd bench: "N cells")
                hits = [l.strip() for l in open(logfile, encoding="utf8", errors="replace") if show in l]
                extra = f" | {hits[-1][-90:]}" if hits else ""
            print(f"{(time.time() - t0) / 60:6.0f} phút: {n_replay() - n0} prompt mới{extra}", flush=True)
    print(f"exit {p.returncode} sau {(time.time() - t0) / 60:.1f} phút", flush=True)
    return p.returncode


In [ ]:
# smoke: 2 câu của 1 shop dev (biz_101, không phải holdout). Lỗi ở đây thì dừng, không chạy tiếp
if RUN_SMOKE:
    assert sh(f"{EVAL} --split dev --only biz_101 --per-shop 2") == 0, "smoke lỗi: gửi output cell này cho lead"

In [ ]:
# chạy chính trên dev
if RUN_DEV:
    t0 = time.time()
    rc = run_live(f"{EVAL} --split dev --per-shop {PER_SHOP}", f"{OUT}/run_eval_dev.txt")
    !grep -vE "^(INFO|WARNING|\[|Processed prompts|Capturing)" {OUT}/run_eval_dev.txt | tail -40
    assert rc == 0, "chạy dev lỗi: gửi run_eval_dev.txt cho lead"
    print(f"{(time.time() - t0) / 60:.1f} phút")

In [ ]:
# calibrate k/m/policy trên trace vừa ghi
if RUN_DEV:
    runs = sorted(glob.glob("eval/runs/*_dev"))
    assert runs, "cell chạy dev ở trên đã lỗi, không có eval/runs/*_dev: gửi output cell đó cho lead"
    DEV_RUN = runs[-1]
    print(DEV_RUN)
    !{PY} eval/calibrate_agent.py {DEV_RUN}/trace.jsonl 2>&1 | tee {OUT}/calibrate_dev.txt
    if AGENT == "sql":
        !{PY} eval/sql_report.py {DEV_RUN}/trace.jsonl 2>&1 | tee {OUT}/sql_report_dev.txt | head -40

In [ ]:
# holdout: chỉ số tổng (không in câu hỏi/lỗi)
if RUN_HOLDOUT:
    t0 = time.time()
    rc = run_live(f"{EVAL} --split holdout --per-shop {HOLDOUT_PER_SHOP}", f"{OUT}/run_eval_holdout.txt")
    !grep -E "^split=" {OUT}/run_eval_holdout.txt
    assert rc == 0, "chạy holdout lỗi: gửi dòng Traceback cuối của run_eval_holdout.txt cho lead"
    HO_RUN = sorted(glob.glob("eval/runs/*_holdout"))[-1]
    if AGENT == "sql":
        !{PY} eval/sql_report.py {HO_RUN}/trace.jsonl --totals-only 2>&1 | tee {OUT}/sql_report_holdout.txt
    !{PY} eval/calibrate_agent.py {HO_RUN}/trace.jsonl --totals-only 2>&1 | tee {OUT}/calibrate_holdout.txt
    print(f"{(time.time() - t0) / 60:.1f} phút")

In [ ]:
# benchmark tổng hợp (E2, docs/BENCHMARK.md): dựng lại biến thể đã niêm phong (CPU) rồi chạy agent trên một side.
# Side B chỉ xem số tổng: chấm điểm (`bench.py score`) làm trên máy lead, không ở đây.
import shutil

if RUN_BENCH:
    # biến thể đã niêm phong: ưu tiên dataset dựng sẵn trên máy lead (datamate-bench-B.zip). Dựng lại ở đây dùng
    # pandas/numpy bản pin của image nên byte file khác bản niêm phong và `run` sẽ từ chối (13/11)
    pre = sorted(glob.glob(f"/kaggle/input/**/{BENCH_SIDE}/rulebooks/rule_param.md", recursive=True))
    dst = f"eval/synth/out/bench/{BENCH_SIDE}"
    if pre:
        src = os.path.dirname(os.path.dirname(pre[0]))
        print("biến thể dựng sẵn:", src)
        shutil.rmtree(dst, ignore_errors=True)
        shutil.copytree(src, dst)
    else:
        assert sh(f"{PY} eval/synth/bench.py materialize --jobs 3") == 0, (
            "materialize lỗi: Add Input dataset datamate-bench-B (biến thể dựng sẵn) rồi chạy lại")
    # kiểm niêm phong TRƯỚC khi nạp model; biến thể lệch được in tên/cỡ/mã băm từng file (gửi lead), không dừng:
    # `run --skip-bad` bỏ riêng các cell của biến thể đó (13/11: B|layout|biz_127 lệch trên Kaggle, khớp trên máy lead)
    if sh(f"{PY} eval/synth/bench.py verify --side {BENCH_SIDE} --fix-names") != 0:
        print("^ có biến thể lệch niêm phong: chụp các dòng trên gửi lead; các cell của chúng sẽ bị bỏ")
    # nối tiếp lượt trước: replay_bench.jsonl của lượt hỏng (Add Input = Output của version đó) -> không sinh lại
    old = sorted(glob.glob("/kaggle/input/**/replay_bench.jsonl", recursive=True))
    if old and not os.path.exists(f"{OUT}/replay_bench.jsonl"):
        shutil.copy(old[-1], f"{OUT}/replay_bench.jsonl")
        print("replay cũ:", old[-1], sum(1 for _ in open(old[-1], encoding="utf8")), "prompt")
    BENCH_DIR = f"eval/runs/bench_{AGENT}_{TAG}_{BENCH_SIDE}"
    rc = run_live(f"{PY} eval/synth/bench.py run --arm {AGENT}_{TAG} --side {BENCH_SIDE} --agent {AGENT} "
                  f"--llm vllm --llm-model {MODEL_DIR} --llm-replay {OUT}/replay_bench.jsonl --out {BENCH_DIR} --skip-bad",
                  f"{OUT}/bench_run.txt", replay=f"{OUT}/replay_bench.jsonl", show=" cells (")
    if rc != 0:
        print("".join(open(f"{OUT}/bench_run.txt", encoding="utf8", errors="replace").readlines()[-25:]))
    assert rc == 0, "bench lỗi: gửi 25 dòng log in ngay trên cho lead"


In [ ]:
# đóng gói: tải file zip này về máy (Output -> Download). KHÔNG commit lên GitHub.
import shutil
for d in sorted(glob.glob("eval/runs/*_dev")) + sorted(glob.glob("eval/runs/*_holdout")) + sorted(glob.glob("eval/runs/bench_*")):
    shutil.copytree(d, f"{OUT}/{os.path.basename(d)}", dirs_exist_ok=True)
!uv pip freeze --python {PY} | grep -iE "^(vllm|torch|transformers|xgrammar)==" > {OUT}/versions.txt
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader >> {OUT}/versions.txt
!git rev-parse --short HEAD >> {OUT}/versions.txt
zipf = shutil.make_archive(f"/kaggle/working/datamate_dev_{AGENT}_{MODEL_NAME}{'_think' if THINKING else ''}_{TAG}", "zip", OUT)
print(zipf, os.path.getsize(zipf) // 1024, "KB")